# Program 16
## Training a Logistic Regression Classifier using Skip-Gram for Word Proximity Prediction
By using Skip-gram, train a logistic regression classifier to compute the probability that two words are 'likely to occur nearby in text'. Use dataset from WSJ corpus.

In [1]:
import nltk
import numpy as np
import random
from collections import Counter
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# Ensure the treebank corpus is downloaded
nltk.download('treebank')
nltk.download('punkt')
from nltk.corpus import treebank

# 1. Load the WSJ corpus and preprocess
print("Loading words from WSJ corpus...")
words = treebank.words()
words = [word.lower() for word in words if word.isalpha()]  # basic filtering

# To keep it computationally feasible, let's take a subset and limit the vocabulary
vocab_size = 2000
word_counts = Counter(words)
top_words = [w for w, count in word_counts.most_common(vocab_size)]
word2idx = {w: i for i, w in enumerate(top_words)}
idx2word = {i: w for w, i in word2idx.items()}

# Filter corpus to only include top words
corpus = [w for w in words if w in word2idx]
print(f"Corpus size after filtering: {len(corpus)}")

# 2. Generate positive and negative samples (Skip-Gram approach)
window_size = 2
X_data = []
y_labels = []

print("Generating context pairs...")
for i in range(len(corpus)):
    target_word = corpus[i]
    target_idx = word2idx[target_word]
    
    # Context window bounds
    start = max(0, i - window_size)
    end = min(len(corpus), i + window_size + 1)
    
    context_words = corpus[start:i] + corpus[i+1:end]
    
    for context_word in context_words:
        context_idx = word2idx[context_word]
        
        # Positive sample (target, context) -> 1
        X_data.append([target_idx, context_idx])
        y_labels.append(1)
        
        # Negative sampling (target, random_word) -> 0
        # Sample a random word from the vocabulary
        random_idx = random.randint(0, vocab_size - 1)
        X_data.append([target_idx, random_idx])
        y_labels.append(0)

X_data = np.array(X_data)
y_labels = np.array(y_labels)

print(f"Generated {len(X_data)} training samples.")

# 3. Feature Extraction (One-Hot Encoding or simple Embeddings)
# For a logistic regression, passing categorical integer indexes doesn't make sense.
# Since one-hot vectors of size 2000 x 2000 is too large for memory,
# We will use feature hashing or create simple embeddings for demonstration.
# Here we'll create manual random embeddings to train the logistic regression.
embedding_dim = 50
np.random.seed(42)
embeddings = np.random.randn(vocab_size, embedding_dim)

def get_features(X_pairs):
    # Concatenate the embeddings of the target word and context word
    return np.hstack((embeddings[X_pairs[:, 0]], embeddings[X_pairs[:, 1]]))

X_features = get_features(X_data)

# 4. Train-test split
X_train, X_test, y_train, y_test = train_test_split(X_features, y_labels, test_size=0.2, random_state=42)

# 5. Train Logistic Regression Classifier
print("Training Logistic Regression Model...")
model = LogisticRegression(max_iter=500, verbose=1)
model.fit(X_train, y_train)

# 6. Evaluate and Predict
y_pred = model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"\nAccuracy on test set: {accuracy * 100:.2f}%")

def predict_proximity(word1, word2):
    if word1 not in word2idx or word2 not in word2idx:
        return "Words not in vocabulary."
    idx1, idx2 = word2idx[word1], word2idx[word2]
    feature = np.hstack((embeddings[idx1], embeddings[idx2])).reshape(1, -1)
    prob = model.predict_proba(feature)[0][1]
    return f"Probability of '{word1}' and '{word2}' occurring nearby: {prob:.4f}"

# Example Usage
print("\n--- Proximity Predictions ---")
print(predict_proximity('wall', 'street'))  # Often together
print(predict_proximity('stock', 'market')) 
print(predict_proximity('dog', 'financial')) # Less likely

[nltk_data] Downloading package treebank to
[nltk_data]     C:\Users\vishn\AppData\Roaming\nltk_data...
[nltk_data]   Package treebank is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\vishn\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Loading words from WSJ corpus...
Corpus size after filtering: 63068
Generating context pairs...
Generated 504532 training samples.
Training Logistic Regression Model...

Accuracy on test set: 59.56%

--- Proximity Predictions ---
Probability of 'wall' and 'street' occurring nearby: 0.5547
Probability of 'stock' and 'market' occurring nearby: 0.5622
Words not in vocabulary.


[Parallel(n_jobs=1)]: Done   1 out of   1 | elapsed:    1.3s finished
